# Core CRUD Workflow


**Workflows covered:**
1. Setup & Engine Connection
2. Read `friends` & `books` tables (+ test after each)
3. Create a friend & loan using dynamically selected row data (+ test)
4. Update a friend using dynamic selection (+ test)
5. Delete a friend using dynamic selection (+ test)


## 0. Setup & Engine Connection

In [7]:
import pandas as pd
from sqlalchemy import create_engine, text

# Running this cell REQUIRES a module named con_lib.py in this notebook's PARENT DIRECTORY
# ALTERNATELY, provide a connection string in a different way
#schema = "Your_Schema"
#host = "127.0.0.1"
##user = "root"
#password = "YOUR_PASSWORD"
##port = 3306

#connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

import sys
sys.path.append("..") # con_lib is the parent directory of this notebook
from con_lib import connection_string

engine = create_engine(connection_string)


In [8]:
pd.read_sql('friends', con=engine)

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso Klein,1,NaN
6,7,Marie,3,NaN
7,8,Olena,5,NaN
8,9,Sayali,3,NaN


In [9]:
pd.read_sql('books', con=connection_string)

,title,author,genre,isbn
0,The Alchemist,Paulo Coelho,Fiction,9780062316110
1,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
5,Echoes of the Past,Julian Marsh,Thriller,9780987654321
6,The Secret Ingredient,Samira Nouri,Romance,9781122334455
7,The Paper Trail,Liane Forestier,Historical Fiction,9781234567890
8,The Clockmaker's Son,Hugo Vernier,Steampunk,9784455667788
9,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899


In [10]:
pd.read_sql('loans', con=connection_string)


,isbn,friend_id,loan_date,last_contact,next_contact,notes
0,9780062316110,2,2026-09-14,NaT,2026-10-14,NaN
1,9780143127741,2,2026-09-14,NaT,2026-10-14,NaN
2,9780987654321,5,2025-07-02,2025-07-02,2025-07-25,NaN
3,9781122334455,3,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
4,9781234567890,1,2025-06-15,2025-06-15,2025-07-15,NaN
5,9784455667788,4,2025-07-01,2025-06-30,2025-07-30,NaN


## 1. Read Operations

In [11]:
def read_friends():
    return pd.read_sql('friends', con=engine)

In [12]:
read_friends()

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso Klein,1,NaN
6,7,Marie,3,NaN
7,8,Olena,5,NaN
8,9,Sayali,3,NaN


In [13]:
def read_books(available_only=False):
    books = pd.read_sql("books", con=engine)
    

    if available_only == True:
        loans= pd.read_sql("loans", con=engine)
        books = pd.merge(books, loans, on='isbn', how='left').query('friend_id.isna()')[books.columns]

    return books

In [14]:
read_books(available_only=True)

,title,author,genre,isbn
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
9,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899


In [15]:
read_books(available_only=True)

,title,author,genre,isbn
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
9,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899


In [16]:
read_books()

,title,author,genre,isbn
0,The Alchemist,Paulo Coelho,Fiction,9780062316110
1,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
5,Echoes of the Past,Julian Marsh,Thriller,9780987654321
6,The Secret Ingredient,Samira Nouri,Romance,9781122334455
7,The Paper Trail,Liane Forestier,Historical Fiction,9781234567890
8,The Clockmaker's Son,Hugo Vernier,Steampunk,9784455667788
9,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899


In [17]:
books = pd.read_sql("books", con=engine)
loans= pd.read_sql("loans", con=engine)

available_books = pd.merge(books, loans, on='isbn', how='left').query('friend_id.isna()')

available_books

,title,author,genre,isbn,friend_id,loan_date,last_contact,next_contact,notes
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486,NaN,NaT,NaT,NaT,NaN
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818,NaN,NaT,NaT,NaT,NaN
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790,NaN,NaT,NaT,NaT,NaN
9,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899,NaN,NaT,NaT,NaT,NaN


In [18]:
read_books()

,title,author,genre,isbn
0,The Alchemist,Paulo Coelho,Fiction,9780062316110
1,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
5,Echoes of the Past,Julian Marsh,Thriller,9780987654321
6,The Secret Ingredient,Samira Nouri,Romance,9781122334455
7,The Paper Trail,Liane Forestier,Historical Fiction,9781234567890
8,The Clockmaker's Son,Hugo Vernier,Steampunk,9784455667788
9,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899


In [19]:
def read_loans():
    return pd.read_sql("loans", con=engine)

In [20]:
read_loans()

,isbn,friend_id,loan_date,last_contact,next_contact,notes
0,9780062316110,2,2026-09-14,NaT,2026-10-14,NaN
1,9780143127741,2,2026-09-14,NaT,2026-10-14,NaN
2,9780987654321,5,2025-07-02,2025-07-02,2025-07-25,NaN
3,9781122334455,3,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
4,9781234567890,1,2025-06-15,2025-06-15,2025-07-15,NaN
5,9784455667788,4,2025-07-01,2025-06-30,2025-07-30,NaN


## 2. Create friend/Loan


In [21]:
engine = create_engine(connection_string)

update_query= """
 INSERT INTO friends(`name`, max_loans)
 VALUES("Marie", 3);
"""

with engine.connect() as connection:
    transaction = connection.begin()
    try:                                          # Python will attempt to run the code inside this block.
        connection.execute(text(update_query))    # sends your raw SQL string (safely wrapped in the text() fucntion) to the database to be executed.
        transaction.commit()                      # If the execution is successful, this line tells the database to save the changes permanently. The transaction is now commplete!
    except:                                       # If any error occurs in the try block, Python immediately jumps here
        transaction.rollback()                    # This is the panic button. It tells the database to instantly undo any temporary changes made since connection.begin().
        raise

In [22]:
def create_friend(name, max_loans=3, notes=None):
    engine = create_engine(connection_string)

    update_query= """
        INSERT INTO friends(`name`, max_loans)
        VALUES(:fname, :max_loans);
    """

    with engine.connect() as connection:
        transaction = connection.begin()
        try:                                          # Python will attempt to run the code inside this block.
            connection.execute(text(update_query), {"fname" : name, "max_loans": max_loans})    # sends your raw SQL string (safely wrapped in the text() fucntion) to the database to be executed.
            transaction.commit()                      # If the execution is successful, this line tells the database to save the changes permanently. The transaction is now commplete!
        except:                                       # If any error occurs in the try block, Python immediately jumps here
            transaction.rollback()                    # This is the panic button. It tells the database to instantly undo any temporary changes made since connection.begin().
            raise

In [23]:
create_friend("Olena", 5)

In [24]:
read_friends()

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso Klein,1,NaN
6,7,Marie,3,NaN
7,8,Olena,5,NaN
8,9,Sayali,3,NaN
9,10,Marie,3,NaN


In [25]:
# Example adding things with query 

update_query = """ 
            INSERT INTO FRIENDS (`name`, max_loans)
            VALUES ('Semira', 3);
"""

with engine.connect() as connection:
    transaction = connection.begin()
    try:
        connection.execute(text(update_query))
        transaction.commit()
    except:
        transaction.rollback()
        raise

In [26]:
read_friends()

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso Klein,1,NaN
6,7,Marie,3,NaN
7,8,Olena,5,NaN
8,9,Sayali,3,NaN
9,10,Marie,3,NaN


In [27]:
# Create_friend function using query insinde python

def create_friend(name, max_loans=5, notes=None):

    engine = create_engine(connection_string)
    
    update_query = """
        INSERT INTO friends (`name`, max_loans, notes)
        VALUES (:name, :max_loans, :notes);
    """

    with engine.begin() as connection:
        connection.execute(
            text(update_query),
            {
                "name": name,
                "max_loans": max_loans,
                "notes": notes
            }
        )


In [28]:
read_friends()

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso Klein,1,NaN
6,7,Marie,3,NaN
7,8,Olena,5,NaN
8,9,Sayali,3,NaN
9,10,Marie,3,NaN


In [29]:
### With pandas 
'''
books, friends

friend_name, isbn, loan_date, next_contact, notes

'''

def create_loan(friend_name, isbn, loan_date=None, next_contact=None, notes=None):
    friends_df = read_friends()
    books_df = read_books()

    # Find the friend_id from friends table
    friend = friends_df[friends_df['name'] == friend_name].iloc[0]

    # book_id
    book = books_df[books_df['isbn'] == isbn].iloc[0]

    df = pd.DataFrame([[book['isbn'],
                        friend['friend_id']],
                        loan_date, 
                        next_contact, 
                        notes],
                      columns=['isbn', 
                               'friend_id',
                                 'loan_date', 
                                 'next_contact', 
                                 'notes'])
    df.to_sql('loans',
              if_exists='append',
              con=engine,
              index=False
              )
    return f"{friend_name} borrowed '{book['title']}' and this added to loans table."



In [30]:
friends_df = read_friends()

In [31]:
friend_name = 'Davey'

In [32]:
matching_friends = friends_df[friends_df["name"] == friend_name]

In [33]:
matching_friends

,friend_id,name,max_loans,notes
1,2,Davey,3,Gets recommendations from Ellie


In [34]:
friend = matching_friends.iloc[0]

In [35]:
friend['notes']

'Gets recommendations from Ellie'

In [36]:
loans_df = read_loans()

In [37]:
current_loans = len(loans_df[loans_df["friend_id"] == friend["friend_id"]])

In [38]:
current_loans

2

In [39]:
#### With SQL 

def create_loan(friend_name, isbn, loan_date=None, next_contact=None, notes=None):

    friends_df = read_friends()
    books_df = read_books(available_only=True)
    loans_df = read_loans()

    # 1. LOOKUP: friend
    matching_friends = friends_df[friends_df["name"] == friend_name]
    if len(matching_friends) == 0:
        return f"Error: No friend found with name '{friend_name}'."
    friend = matching_friends.iloc[0]

    # 2. LOOKUP: book
    matching_books = books_df[books_df["isbn"] == isbn]
    if len(matching_books) == 0:
        return f"Error: No book found with isbn '{isbn}'."
    book = matching_books.iloc[0]

    # 3. CHECK MAX LOANS: count current loans for this friend
    current_loans = len(loans_df[loans_df["friend_id"] == friend["friend_id"]])
    if current_loans >= friend["max_loans"]:
        return f"Warning: '{friend['name']}' has reached their maximum loan allowance ({friend['max_loans']})."

    # 4. DATES: default logic
    if loan_date is None:
        loan_date = pd.Timestamp.today().date()
    if next_contact is None:
        next_contact = loan_date + pd.Timedelta(30, "D")

    # 5. INSERT: create loan record using a SQL query
    engine = create_engine(connection_string)

    insert_query = """
        INSERT INTO loans (isbn, friend_id, loan_date, next_contact, notes)
        VALUES (:isbn, :friend_id, :loan_date, :next_contact, :notes);
    """

    with engine.begin() as connection:
        connection.execute(
            text(insert_query),
            {
                "isbn": book["isbn"],
                "friend_id": friend["friend_id"],
                "loan_date": loan_date,
                "next_contact": next_contact,
                "notes": notes
            }
        )

    return f"Added '{friend['name']}' borrowed '{book['title']}' to 'loans'."

In [40]:
read_books(available_only=True)


,title,author,genre,isbn
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
9,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899


In [41]:
create_loan('Davey','9780143127741')

"Error: No book found with isbn '9780143127741'."

In [42]:
read_loans()


,isbn,friend_id,loan_date,last_contact,next_contact,notes
0,9780062316110,2,2026-09-14,NaT,2026-10-14,NaN
1,9780143127741,2,2026-09-14,NaT,2026-10-14,NaN
2,9780987654321,5,2025-07-02,2025-07-02,2025-07-25,NaN
3,9781122334455,3,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
4,9781234567890,1,2025-06-15,2025-06-15,2025-07-15,NaN
5,9784455667788,4,2025-07-01,2025-06-30,2025-07-30,NaN


In [43]:
read_books(available_only=True)

,title,author,genre,isbn
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
9,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899


In [44]:
read_loans()

,isbn,friend_id,loan_date,last_contact,next_contact,notes
0,9780062316110,2,2026-09-14,NaT,2026-10-14,NaN
1,9780143127741,2,2026-09-14,NaT,2026-10-14,NaN
2,9780987654321,5,2025-07-02,2025-07-02,2025-07-25,NaN
3,9781122334455,3,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
4,9781234567890,1,2025-06-15,2025-06-15,2025-07-15,NaN
5,9784455667788,4,2025-07-01,2025-06-30,2025-07-30,NaN


In [45]:
create_loan("Semira", "9785566778899")

"Added 'Semira' borrowed 'Gardens of Glass' to 'loans'."

## 3. Update Friend

In [46]:
def update_friend(friend_name, field, new_data):

    friends_df = read_friends()

    matching_friends = friends_df[friends_df["name"] == friend_name]
    if len(matching_friends) == 0:
        return f"Warning: '{friend_name}' not found."

    friend_id = matching_friends.iloc[0]["friend_id"]

    engine = create_engine(connection_string)

    query = f"""
        UPDATE friends
        SET {field} = :val
        WHERE friend_id = :id;
    """

    with engine.begin() as connection:
        connection.execute(
            text(query),
            {"val": new_data, "id": friend_id}
        )

    return f"'{field}' has been updated for '{friend_name}'."

In [47]:
update_friend('Davey', 'notes', 'we dont like him anymore')

"'notes' has been updated for 'Davey'."

## 4. Delete Friend

In [49]:
read_friends()

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,we dont like him anymore
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso Klein,1,NaN
6,7,Marie,3,NaN
7,8,Olena,5,NaN
8,9,Sayali,3,NaN
9,10,Marie,3,NaN


In [50]:
def delete_friend(friend_name):

    friends_df = read_friends()

    matching_friends = friends_df[friends_df["name"] == friend_name]
    if len(matching_friends) == 0:
        return f"Warning: '{friend_name}' doesn't exist in friends list."

    friend_id = matching_friends.iloc[0]["friend_id"]

    engine = create_engine(connection_string)

    query = """
        DELETE FROM friends
        WHERE friend_id = :id;
    """

    with engine.begin() as connection:
        connection.execute(
            text(query),
            {"id": friend_id}
        )

    return f"'{friend_name}' has been deleted."

In [51]:
delete_friend("Semira")

"'Semira' has been deleted."

In [52]:
read_friends()

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,we dont like him anymore
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso Klein,1,NaN
6,7,Marie,3,NaN
7,8,Olena,5,NaN
8,9,Sayali,3,NaN
9,10,Marie,3,NaN


In [53]:
delete_friend('Marie')

"'Marie' has been deleted."

In [54]:
read_friends()

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,we dont like him anymore
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso Klein,1,NaN
6,8,Olena,5,NaN
7,9,Sayali,3,NaN
8,10,Marie,3,NaN
9,11,Olena,5,NaN
